In [1]:
import re
import time
from collections import Counter, defaultdict

import pandas as pd

cols = ["entity_id", "country", "name_core", "addr_clean"]

s1 = pd.read_csv("s1_clean.tsv", sep="\t", dtype=str, keep_default_na=False, usecols=cols)
s2 = pd.read_csv("s2_clean.tsv", sep="\t", dtype=str, keep_default_na=False, usecols=cols)
s3 = pd.read_csv("s3_clean.tsv", sep="\t", dtype=str, keep_default_na=False, usecols=cols)

data_path = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train"
gt = pd.read_csv(data_path + r"\train_ground_truth.tsv", sep="\t", dtype=str)

print("s1:", s1.shape, " s2:", s2.shape, " s3:", s3.shape, " gt:", gt.shape)
print(s1.head(3))

s1: (2206821, 4)  s2: (5034616, 4)  s3: (5285603, 4)  gt: (2206821, 2)
      entity_id country                      addr_clean           name_core
0  S1-925783039      US  1795 westchester dr high pt nc  orelees barbershop
1  S1-773889195      US     17560 ellis rd tahlequah ok         prime money
2  S1-377745466      US  1712 montebello ave phoenix az            b retail


In [2]:
print(s2.head(3))

      entity_id country                         addr_clean  \
0  S2-166376419   India        kh no 570 13 new dl w dl dl   
1  S2-764573417      US            105 elm st morganton nc   
2  S2-639257739   India  g 3 571 gulmohar colony bhopal mp   

               name_core  
0          ram marketing  
1  holloway peak seafood  
2      aditya properties  


In [3]:
# one row per matched pair
pairs = gt.dropna(subset=["matched_entity_ids"]).copy()
pairs["match_id"] = pairs["matched_entity_ids"].str.split(",")
pairs = pairs.explode("match_id")
pairs["match_id"] = pairs["match_id"].str.strip()

# country of every record
country_of = pd.concat([s1[["entity_id", "country"]],
                        s2[["entity_id", "country"]],
                        s3[["entity_id", "country"]]]).set_index("entity_id")["country"]

pairs["country_s1"] = pairs["source1_entity_id"].map(country_of)
pairs["country_other"] = pairs["match_id"].map(country_of)

same = (pairs["country_s1"] == pairs["country_other"])
print("matched pairs:", len(pairs))
print("same country:     ", same.sum(), f"({100 * same.mean():.3f}%)")
print("different country:", (~same).sum())
print(pairs.loc[~same, ["country_s1", "country_other"]].value_counts().head(10))

# S1 records with no match
n_single = gt["matched_entity_ids"].isna().sum()
print(f"\nS1 with no match: {n_single} of {len(gt)} ({100 * n_single / len(gt):.1f}%)")

# records per country
for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
    print(name, df["country"].value_counts().to_dict())

matched pairs: 7638365
same country:      7638365 (100.000%)
different country: 0
Series([], Name: count, dtype: int64)

S1 with no match: 123247 of 2206821 (5.6%)
s1 {'US': 1323633, 'India': 883188}
s2 {'US': 3016817, 'India': 2017799}
s3 {'US': 3170056, 'India': 2115547}


In [4]:
pip install jellyfish

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Hp\OneDrive\Desktop\amazon hackthon\amazon\Scripts\python.exe -m pip install --upgrade pip' command.


In [5]:
import jellyfish

# 1. every different word in all names (the vocabulary)
vocab = set()
for df in [s1, s2, s3]:
    for name in df["name_core"]:
        vocab.update(name.split())
print("different words:", len(vocab))

# 2. soundex code for each word, computed once
def word_code(w):
    if any(ch.isdigit() for ch in w):     # numbers like "3m" or "24x7" stay as they are
        return w
    return jellyfish.soundex(w)

code_of = {w: word_code(w) for w in vocab}

# 3. key = codes of all words, sorted
def sound_key(name):
    return " ".join(sorted(code_of[w] for w in name.split()))

for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
    start = time.time()
    df["key_a"] = df["name_core"].map(sound_key)
    print(name, "done in", round(time.time() - start), "seconds")

print(s1[["name_core", "key_a"]].head(10).to_string())

different words: 1433074
s1 done in 9 seconds
s2 done in 25 seconds
s3 done in 28 seconds
                    name_core           key_a
0          orelees barbershop       B616 O642
1                 prime money       M500 P650
2                    b retail       B000 R340
3               christ chapel       C140 C623
4     prabhav business center  B252 C536 P611
5      custom wealth services  C235 S612 W430
6    consulting nyasa nursing  C524 N200 N625
7           nexus anchor rain  A526 N220 R500
8               moore bitwise       B320 M600
9  dermatology green medicine  D653 G650 M325


In [6]:
MAX_BUCKET = 1000     # buckets bigger than this are skipped

# key of every record
key_of = pd.concat([s1[["entity_id", "key_a"]],
                    s2[["entity_id", "key_a"]],
                    s3[["entity_id", "key_a"]]]).set_index("entity_id")["key_a"]

pairs["key_s1"] = pairs["source1_entity_id"].map(key_of)
pairs["key_other"] = pairs["match_id"].map(key_of)

# size of each bucket = how many S2+S3 records share (country, key)
others = pd.concat([s2[["country", "key_a"]], s3[["country", "key_a"]]])
bucket_size = others.groupby(["country", "key_a"]).size().rename("bucket_size")

pairs = pairs.drop(columns=["bucket_size"], errors="ignore")
pairs = pairs.merge(bucket_size, left_on=["country_s1", "key_s1"], right_index=True, how="left")
pairs["bucket_size"] = pairs["bucket_size"].fillna(0)

# a true match is FOUND if both have the same key, and the bucket isn't too big
found_a = ((pairs["key_s1"] == pairs["key_other"]) &
           (pairs["key_s1"] != "") &
           (pairs["bucket_size"] <= MAX_BUCKET))
pairs["found_a"] = found_a
print(f"Key A recall: {100 * found_a.mean():.2f}% of true matches")

# how many candidates each S1 gets
s1c = s1.merge(bucket_size, left_on=["country", "key_a"], right_index=True, how="left")
s1c["bucket_size"] = s1c["bucket_size"].fillna(0)
s1c.loc[(s1c["bucket_size"] > MAX_BUCKET) | (s1c["key_a"] == ""), "bucket_size"] = 0
print(f"candidates per S1: average {s1c['bucket_size'].mean():.1f}, "
      f"median {s1c['bucket_size'].median():.0f}, max {s1c['bucket_size'].max():.0f}")
print(f"total candidate pairs: {int(s1c['bucket_size'].sum()):,}")

Key A recall: 58.96% of true matches
candidates per S1: average 80.5, median 7, max 1000
total candidate pairs: 177,675,386


In [7]:
for limit in [50, 100, 200, 500, 1000]:
    found = ((pairs["key_s1"] == pairs["key_other"]) &
             (pairs["key_s1"] != "") &
             (pairs["bucket_size"] <= limit))
    size = s1c["bucket_size"].where(s1c["bucket_size"] <= limit, 0)
    print(f"limit {limit:5d}:  recall {100 * found.mean():5.2f}%   "
          f"candidates per S1 {size.mean():6.1f}   total pairs {int(size.sum()):>13,}")

limit    50:  recall 43.09%   candidates per S1    5.6   total pairs    12,313,535
limit   100:  recall 44.82%   candidates per S1    7.7   total pairs    16,903,543
limit   200:  recall 51.15%   candidates per S1   23.1   total pairs    50,924,703
limit   500:  recall 56.25%   candidates per S1   50.3   total pairs   110,907,310
limit  1000:  recall 58.96%   candidates per S1   80.5   total pairs   177,675,386


In [8]:
name_of = pd.concat([s1[["entity_id", "name_core"]],
                     s2[["entity_id", "name_core"]],
                     s3[["entity_id", "name_core"]]]).set_index("entity_id")["name_core"]

missed = pairs[~pairs["found_a"]].sample(25, random_state=7)
missed = missed.assign(name_s1=missed["source1_entity_id"].map(name_of),
                       name_other=missed["match_id"].map(name_of))
print(missed[["name_s1", "name_other", "key_s1", "key_other", "bucket_size"]].to_string())

                                    name_s1                                  name_other                    key_s1                      key_other  bucket_size
507651                                 doex                                       doiex                      D200                           D200       1317.0
134102                    royal select east                            royal select esa            E230 R400 S423                 E200 R400 S423          2.0
1734069                         links diner                              linksdiner com                 D560 L520                      C500 L523          4.0
1022192                     heartland value                         the heartland value                 H634 V400                 H634 T000 V400         20.0
728155                        royal finance            deltapyra formerly royal finance                 F552 R400            D431 F552 F656 R400        285.0
861818                    aditesh solutions         

In [9]:
stop_words = {"the", "and", "of", "com", "www", "for", "in", "at"}

# how many records contain each word (all sources)
word_count = Counter()
for df in [s1, s2, s3]:
    for name in df["name_core"]:
        word_count.update(set(name.split()))
print("different words:", len(word_count))

def rare_words(name):
    words = [w for w in set(name.split())
             if len(w) >= 3 and not w.isdigit() and w not in stop_words]
    words.sort(key=lambda w: (word_count[w], w))     # rarest first
    return words[:3]

for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
    start = time.time()
    rw = df["name_core"].map(rare_words)
    for i in range(3):
        df[f"key_b{i + 1}"] = rw.str[i].fillna("")
    print(name, "done in", round(time.time() - start), "seconds")

print(s1[["name_core", "key_b1", "key_b2"]].head(10).to_string())

different words: 1433074
s1 done in 40 seconds
s2 done in 72 seconds
s3 done in 83 seconds
                    name_core       key_b1      key_b2
0          orelees barbershop      orelees  barbershop
1                 prime money        money       prime
2                    b retail       retail            
3               christ chapel       chapel      christ
4     prabhav business center      prabhav    business
5      custom wealth services       wealth      custom
6    consulting nyasa nursing        nyasa     nursing
7           nexus anchor rain         rain       nexus
8               moore bitwise      bitwise       moore
9  dermatology green medicine  dermatology       green


In [10]:
MAX_BUCKET = 200
other_cols = ["key_b1", "key_b2", "key_b3"]    # S2/S3 use 3 rare words
s1_cols = ["key_b1", "key_b2"]                 # S1 uses 2

# bucket size = how many S2+S3 records have this word as a rare word (per country)
long = pd.concat([pd.DataFrame({"country": df["country"], "word": df[c]})
                  for df in [s2, s3] for c in other_cols])
long = long[long["word"] != ""]
size_b = long.groupby(["country", "word"]).size().to_dict()
del long

# rare words of both records in each true pair
b_of = pd.concat([df[["entity_id"] + other_cols] for df in [s1, s2, s3]]).set_index("entity_id")
for c in s1_cols:
    pairs[c + "_s1"] = pairs["source1_entity_id"].map(b_of[c])
for c in other_cols:
    pairs[c + "_other"] = pairs["match_id"].map(b_of[c])

# found if an S1 rare word is among the other record's rare words, and bucket <= limit
found_b = pd.Series(False, index=pairs.index)
for c in s1_cols:
    w = pairs[c + "_s1"]
    shared = (w != "") & (w.eq(pairs["key_b1_other"]) |
                          w.eq(pairs["key_b2_other"]) |
                          w.eq(pairs["key_b3_other"]))
    size = pd.Series(list(zip(pairs["country_s1"], w)), index=pairs.index).map(size_b).fillna(0)
    found_b |= shared & (size <= MAX_BUCKET)
pairs["found_b"] = found_b

# Key A again with the new limit of 200
pairs["found_a"] = ((pairs["key_s1"] == pairs["key_other"]) &
                    (pairs["key_s1"] != "") &
                    (pairs["bucket_size"] <= MAX_BUCKET))

print(f"Key A recall:      {100 * pairs['found_a'].mean():.2f}%")
print(f"Key B recall:      {100 * pairs['found_b'].mean():.2f}%")
print(f"A or B together:   {100 * (pairs['found_a'] | pairs['found_b']).mean():.2f}%")

# candidates per S1 from Key B
cand_b = pd.Series(0.0, index=s1.index)
for c in s1_cols:
    size = pd.Series(list(zip(s1["country"], s1[c])), index=s1.index).map(size_b).fillna(0)
    cand_b += size.where(size <= MAX_BUCKET, 0)
print(f"Key B candidates per S1: average {cand_b.mean():.1f}, total {int(cand_b.sum()):,}")

Key A recall:      51.15%
Key B recall:      29.74%
A or B together:   61.83%
Key B candidates per S1: average 30.4, total 67,052,758


In [11]:
addr_of = pd.concat([s1[["entity_id", "addr_clean"]],
                     s2[["entity_id", "addr_clean"]],
                     s3[["entity_id", "addr_clean"]]]).set_index("entity_id")["addr_clean"]

missed_ab = pairs[~(pairs["found_a"] | pairs["found_b"])].sample(25, random_state=11)
show = pd.DataFrame({
    "name_s1":    missed_ab["source1_entity_id"].map(name_of),
    "name_other": missed_ab["match_id"].map(name_of),
    "addr_s1":    missed_ab["source1_entity_id"].map(addr_of),
    "addr_other": missed_ab["match_id"].map(addr_of),
})
pd.set_option("display.max_colwidth", 60)
print(show.to_string())

                                        name_s1                                      name_other                                                                                            addr_s1                                                                              addr_other
911131     pediatric dentistry group of amherst             pediatric dentistry group of center                                                                          1418 french rd amherst ny                                                               1418 french rd amherst ny
1420896                    martin partners care                          martin poarntders care                                                                      6011 princeton st amarillo tx                                                           6011 princeton st amarillo tx
64210                          automation green                            smt automation grgen                                                        

In [12]:
def add_rare_keys(text_col, prefix, stop_words=set()):
    # count how many records contain each word
    counts = Counter()
    for df in [s1, s2, s3]:
        for text in df[text_col]:
            counts.update(set(text.split()))

    def rare(text):
        words = [w for w in set(text.split())
                 if len(w) >= 3 and not w.isdigit() and w not in stop_words]
        words.sort(key=lambda w: (counts[w], w))      # rarest first
        return words[:3]

    for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
        start = time.time()
        rw = df[text_col].map(rare)
        for i in range(3):
            df[f"{prefix}{i + 1}"] = rw.str[i].fillna("")
        print(name, "done in", round(time.time() - start), "seconds")


def found_by_rare_keys(prefix):
    other_cols = [f"{prefix}1", f"{prefix}2", f"{prefix}3"]   # S2/S3 keep 3
    s1_cols = [f"{prefix}1", f"{prefix}2"]                    # S1 keeps 2

    # bucket sizes: how many S2+S3 records have this word, per country
    long = pd.concat([pd.DataFrame({"country": df["country"], "word": df[c]})
                      for df in [s2, s3] for c in other_cols])
    long = long[long["word"] != ""]
    size = long.groupby(["country", "word"]).size().to_dict()
    del long

    key_of = pd.concat([df[["entity_id"] + other_cols] for df in [s1, s2, s3]]).set_index("entity_id")
    other = {c: pairs["match_id"].map(key_of[c]) for c in other_cols}

    # found = an S1 word is among the other record's words, and bucket <= limit
    found = pd.Series(False, index=pairs.index)
    for c in s1_cols:
        w = pairs["source1_entity_id"].map(key_of[c])
        shared = (w != "") & (w.eq(other[other_cols[0]]) |
                              w.eq(other[other_cols[1]]) |
                              w.eq(other[other_cols[2]]))
        bsize = pd.Series(list(zip(pairs["country_s1"], w)), index=pairs.index).map(size).fillna(0)
        found |= shared & (bsize <= MAX_BUCKET)

    # candidates per S1
    cand = pd.Series(0.0, index=s1.index)
    for c in s1_cols:
        bsize = pd.Series(list(zip(s1["country"], s1[c])), index=s1.index).map(size).fillna(0)
        cand += bsize.where(bsize <= MAX_BUCKET, 0)
    return found, cand

In [13]:
add_rare_keys("addr_clean", "key_c")

found_c, cand_c = found_by_rare_keys("key_c")
pairs["found_c"] = found_c

print(f"Key C recall:          {100 * found_c.mean():.2f}%")
print(f"A or B:                {100 * (pairs['found_a'] | pairs['found_b']).mean():.2f}%")
print(f"A or B or C together:  {100 * (pairs['found_a'] | pairs['found_b'] | pairs['found_c']).mean():.2f}%")
print(f"Key C candidates per S1: average {cand_c.mean():.1f}, total {int(cand_c.sum()):,}")

s1 done in 22 seconds
s2 done in 44 seconds
s3 done in 50 seconds
Key C recall:          36.09%
A or B:                61.83%
A or B or C together:  75.99%
Key C candidates per S1: average 30.7, total 67,639,377


In [14]:
def add_rare_keys(text_col, prefix, stop_words=set(), min_count=2):
    counts = Counter()
    for df in [s1, s2, s3]:
        for text in df[text_col]:
            counts.update(set(text.split()))

    def rare(text):
        words = [w for w in set(text.split())
                 if len(w) >= 3 and not w.isdigit() and w not in stop_words
                 and counts[w] >= min_count]              # NEW: skip words seen only once
        words.sort(key=lambda w: (counts[w], w))
        return words[:3]

    for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
        start = time.time()
        rw = df[text_col].map(rare)
        for i in range(3):
            df[f"{prefix}{i + 1}"] = rw.str[i].fillna("")
        print(name, "done in", round(time.time() - start), "seconds")

In [15]:
stop_words = {"the", "and", "of", "com", "www", "for", "in", "at"}

add_rare_keys("name_core", "key_b", stop_words=stop_words)
found_b, cand_b = found_by_rare_keys("key_b")
pairs["found_b"] = found_b

add_rare_keys("addr_clean", "key_c")
found_c, cand_c = found_by_rare_keys("key_c")
pairs["found_c"] = found_c

print(f"Key B recall:          {100 * found_b.mean():.2f}%   (before 29.74%)")
print(f"Key C recall:          {100 * found_c.mean():.2f}%   (before 36.09%)")
print(f"A or B or C together:  {100 * (pairs['found_a'] | pairs['found_b'] | pairs['found_c']).mean():.2f}%   (before 75.99%)")
print(f"candidates per S1: B {cand_b.mean():.1f}, C {cand_c.mean():.1f}")

s1 done in 13 seconds
s2 done in 44 seconds
s3 done in 44 seconds
s1 done in 39 seconds
s2 done in 110 seconds
s3 done in 119 seconds
Key B recall:          29.74%   (before 29.74%)
Key C recall:          36.08%   (before 36.09%)
A or B or C together:  75.99%   (before 75.99%)
candidates per S1: B 30.4, C 30.7


In [16]:
import numpy as np

def smallest_shared_bucket(prefix):
    # for each true pair: size of the smallest bucket they share (inf = share nothing)
    other_cols = [f"{prefix}1", f"{prefix}2", f"{prefix}3"]
    s1_cols = [f"{prefix}1", f"{prefix}2"]

    long = pd.concat([pd.DataFrame({"country": df["country"], "word": df[c]})
                      for df in [s2, s3] for c in other_cols])
    long = long[long["word"] != ""]
    size = long.groupby(["country", "word"]).size().to_dict()
    del long

    key_of = pd.concat([df[["entity_id"] + other_cols] for df in [s1, s2, s3]]).set_index("entity_id")
    other = {c: pairs["match_id"].map(key_of[c]) for c in other_cols}

    best = pd.Series(np.inf, index=pairs.index)
    for c in s1_cols:
        w = pairs["source1_entity_id"].map(key_of[c])
        shared = (w != "") & (w.eq(other[other_cols[0]]) |
                              w.eq(other[other_cols[1]]) |
                              w.eq(other[other_cols[2]]))
        bsize = pd.Series(list(zip(pairs["country_s1"], w)), index=pairs.index).map(size).fillna(np.inf)
        best = best.where(~shared, np.minimum(best, bsize))
    return best

best_b = smallest_shared_bucket("key_b")
best_c = smallest_shared_bucket("key_c")

In [17]:
found_a = pairs["found_a"]

print(f"{'limit':>8} | {'Key B':>7} | {'Key C':>7} | {'A or B or C':>11}")
for limit in [200, 500, 1000, 5000, np.inf]:
    b = best_b <= limit
    c = best_c <= limit
    print(f"{str(limit):>8} | {100 * b.mean():6.2f}% | {100 * c.mean():6.2f}% | {100 * (found_a | b | c).mean():10.2f}%")

print(f"\nshare NO kept word at all:  B {100 * np.isinf(best_b).mean():.1f}%   C {100 * np.isinf(best_c).mean():.1f}%")

   limit |   Key B |   Key C | A or B or C
     200 |  29.74% |  36.08% |      75.99%
     500 |  37.17% |  47.75% |      81.77%
    1000 |  41.12% |  58.33% |      86.03%
    5000 |  56.51% |  79.94% |      94.22%
     inf | 100.00% | 100.00% |     100.00%

share NO kept word at all:  B 11.4%   C 10.6%


In [18]:
from pandas.util import hash_array

# house number = first number in the address
for df in [s1, s2, s3]:
    df["house_no"] = df["addr_clean"].str.extract(r"\b(\d+)\b")[0].fillna("")

def compound_hash(df, parts):
    # joins country + parts into one key, then turns it into a number (saves memory)
    key = df["country"].astype(str)
    empty = pd.Series(False, index=df.index)
    for p in parts:
        key = key + "|" + df[p]
        empty |= (df[p] == "")
    h = hash_array(key.to_numpy(dtype=object)).astype(np.int64)
    h[empty.to_numpy()] = 0          # 0 = no key (a part was missing)
    return h

def smallest_shared_compound(name, s1_combos, other_combos):
    # make the key columns
    for df, combos in [(s1, s1_combos), (s2, other_combos), (s3, other_combos)]:
        for i, parts in enumerate(combos):
            df[f"{name}_{i}"] = compound_hash(df, parts)
    other_cols = [f"{name}_{i}" for i in range(len(other_combos))]
    s1_cols = [f"{name}_{i}" for i in range(len(s1_combos))]

    # bucket sizes
    all_h = np.concatenate([df[c].to_numpy() for df in [s2, s3] for c in other_cols])
    all_h = all_h[all_h != 0]
    size = pd.Series(all_h).value_counts()
    del all_h

    # smallest shared bucket for each true pair
    s1_idx = s1.set_index("entity_id")
    oth_idx = pd.concat([s2[["entity_id"] + other_cols], s3[["entity_id"] + other_cols]]).set_index("entity_id")
    other = {c: pairs["match_id"].map(oth_idx[c]).fillna(0).astype(np.int64) for c in other_cols}

    best = pd.Series(np.inf, index=pairs.index)
    for c in s1_cols:
        w = pairs["source1_entity_id"].map(s1_idx[c]).fillna(0).astype(np.int64)
        match_any = pd.Series(False, index=pairs.index)
        for oc in other_cols:
            match_any |= w.eq(other[oc])
        shared = (w != 0) & match_any
        bsize = w.map(size).fillna(np.inf)
        best = best.where(~shared, np.minimum(best, bsize))
    return best

In [19]:
# D: house number + rare address word
best_d = smallest_shared_compound(
    "key_d",
    s1_combos=[["house_no", "key_c1"], ["house_no", "key_c2"]],
    other_combos=[["house_no", "key_c1"], ["house_no", "key_c2"], ["house_no", "key_c3"]])

# E: rare name word + rare address word
best_e = smallest_shared_compound(
    "key_e",
    s1_combos=[["key_b1", "key_c1"], ["key_b1", "key_c2"], ["key_b2", "key_c1"], ["key_b2", "key_c2"]],
    other_combos=[["key_b1", "key_c1"], ["key_b1", "key_c2"], ["key_b2", "key_c1"],
                  ["key_b2", "key_c2"], ["key_b3", "key_c1"], ["key_b3", "key_c2"]])

print(f"{'limit':>6} | {'D':>7} | {'E':>7} | {'A+B+C+D+E':>10}")
for limit in [50, 100, 200, 500]:
    b, c = best_b <= limit, best_c <= limit
    d, e = best_d <= limit, best_e <= limit
    all_keys = found_a | b | c | d | e
    print(f"{limit:>6} | {100*d.mean():6.2f}% | {100*e.mean():6.2f}% | {100*all_keys.mean():9.2f}%")

 limit |       D |       E |  A+B+C+D+E
    50 |  61.61% |  78.11% |     92.98%
   100 |  62.26% |  78.53% |     93.68%
   200 |  62.58% |  78.66% |     94.39%
   500 |  62.76% |  78.71% |     95.10%


In [20]:
def cand_rare(prefix, limit):
    other_cols = [f"{prefix}1", f"{prefix}2", f"{prefix}3"]
    long = pd.concat([pd.DataFrame({"country": df["country"], "word": df[c]})
                      for df in [s2, s3] for c in other_cols])
    long = long[long["word"] != ""]
    size = long.groupby(["country", "word"]).size().to_dict()
    del long
    total = 0
    for c in [f"{prefix}1", f"{prefix}2"]:
        b = pd.Series(list(zip(s1["country"], s1[c]))).map(size).fillna(0)
        total += b.where(b <= limit, 0).sum()
    return total

def cand_compound(name, n_s1, n_other, limit):
    all_h = np.concatenate([df[f"{name}_{i}"].to_numpy() for df in [s2, s3] for i in range(n_other)])
    size = pd.Series(all_h[all_h != 0]).value_counts()
    del all_h
    total = 0
    for i in range(n_s1):
        h = s1[f"{name}_{i}"]
        b = h.map(size).fillna(0)
        b[h == 0] = 0
        total += b.where(b <= limit, 0).sum()
    return total

for limit in [50, 200]:
    a = s1c["bucket_size"].where(s1c["bucket_size"] <= limit, 0).sum()
    print(f"\nlimit {limit}: candidate pairs per key (millions)")
    print(f"  A {a/1e6:6.1f}   B {cand_rare('key_b', limit)/1e6:6.1f}   C {cand_rare('key_c', limit)/1e6:6.1f}   "
          f"D {cand_compound('key_d', 2, 3, limit)/1e6:6.1f}   E {cand_compound('key_e', 4, 6, limit)/1e6:6.1f}")

    fa = (pairs["key_s1"] == pairs["key_other"]) & (pairs["key_s1"] != "") & (pairs["bucket_size"] <= limit)
    b, c = best_b <= limit, best_c <= limit
    d, e = best_d <= limit, best_e <= limit
    print(f"  recall  A+D+E: {100*(fa|d|e).mean():.2f}%   A+C+D+E: {100*(fa|c|d|e).mean():.2f}%   "
          f"A+B+D+E: {100*(fa|b|d|e).mean():.2f}%   ALL: {100*(fa|b|c|d|e).mean():.2f}%")


limit 50: candidate pairs per key (millions)
  A   12.3   B    8.1   C   12.0   D   24.4   E   52.8
  recall  A+D+E: 90.22%   A+C+D+E: 91.34%   A+B+D+E: 90.74%   ALL: 91.86%

limit 200: candidate pairs per key (millions)
  A   50.9   B   67.1   C   67.7   D   37.0   E   82.6
  recall  A+D+E: 91.68%   A+C+D+E: 93.27%   A+B+D+E: 92.80%   ALL: 94.39%


In [21]:
import jellyfish

def make_sound_column(text_col, new_col):
    vocab = set()
    for df in [s1, s2, s3]:
        for text in df[text_col]:
            vocab.update(text.split())
    code = {w: (w if any(ch.isdigit() for ch in w) else jellyfish.metaphone(w)) for w in vocab}
    print(text_col, "different words:", len(vocab))
    for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
        start = time.time()
        df[new_col] = df[text_col].map(lambda t: " ".join(code[w] for w in t.split()))
        print("  ", name, "done in", round(time.time() - start), "seconds")

make_sound_column("name_core", "name_sound")
make_sound_column("addr_clean", "addr_sound")

# single-word sound keys (same functions as B and C)
add_rare_keys("name_sound", "key_bs")
add_rare_keys("addr_sound", "key_cs")
best_bs = smallest_shared_bucket("key_bs")
best_cs = smallest_shared_bucket("key_cs")

# combined sound keys (same function as D and E)
best_ds = smallest_shared_compound(
    "key_ds",
    s1_combos=[["house_no", "key_cs1"], ["house_no", "key_cs2"]],
    other_combos=[["house_no", "key_cs1"], ["house_no", "key_cs2"], ["house_no", "key_cs3"]])

best_es = smallest_shared_compound(
    "key_es",
    s1_combos=[["key_bs1", "key_cs1"], ["key_bs1", "key_cs2"], ["key_bs2", "key_cs1"], ["key_bs2", "key_cs2"]],
    other_combos=[["key_bs1", "key_cs1"], ["key_bs1", "key_cs2"], ["key_bs2", "key_cs1"],
                  ["key_bs2", "key_cs2"], ["key_bs3", "key_cs1"], ["key_bs3", "key_cs2"]])
print("done!")

name_core different words: 1433074
   s1 done in 10 seconds
   s2 done in 17 seconds
   s3 done in 17 seconds
addr_clean different words: 744345
   s1 done in 12 seconds
   s2 done in 26 seconds
   s3 done in 27 seconds
s1 done in 50 seconds
s2 done in 33 seconds
s3 done in 37 seconds
s1 done in 22 seconds
s2 done in 46 seconds
s3 done in 47 seconds
done!


In [22]:
for limit in [50, 200]:
    fa = (pairs["key_s1"] == pairs["key_other"]) & (pairs["key_s1"] != "") & (pairs["bucket_size"] <= limit)
    b, c, d, e = best_b <= limit, best_c <= limit, best_d <= limit, best_e <= limit
    bs, cs, ds, es = best_bs <= limit, best_cs <= limit, best_ds <= limit, best_es <= limit

    word_keys  = fa | b | c | d | e
    sound_keys = fa | bs | cs | ds | es
    both       = word_keys | bs | cs | ds | es

    print(f"\nlimit {limit}")
    print(f"  single keys:  Bs {100*bs.mean():.1f}%  Cs {100*cs.mean():.1f}%  "
          f"Ds {100*ds.mean():.1f}%  Es {100*es.mean():.1f}%")
    print(f"  A + word keys (B,C,D,E):       {100*word_keys.mean():.2f}%")
    print(f"  A + sound keys (Bs,Cs,Ds,Es):  {100*sound_keys.mean():.2f}%")
    print(f"  A + word + sound keys:         {100*both.mean():.2f}%")


limit 50
  single keys:  Bs 3.6%  Cs 11.8%  Ds 60.0%  Es 73.5%
  A + word keys (B,C,D,E):       91.86%
  A + sound keys (Bs,Cs,Ds,Es):  89.65%
  A + word + sound keys:         92.58%

limit 200
  single keys:  Bs 10.8%  Cs 21.4%  Ds 61.8%  Es 75.7%
  A + word keys (B,C,D,E):       94.39%
  A + sound keys (Bs,Cs,Ds,Es):  92.50%
  A + word + sound keys:         94.91%


In [23]:
limit = 50
fa = (pairs["key_s1"] == pairs["key_other"]) & (pairs["key_s1"] != "") & (pairs["bucket_size"] <= limit)
word_keys = fa | (best_b <= limit) | (best_c <= limit) | (best_d <= limit) | (best_e <= limit)
print(f"word keys: {100 * word_keys.mean():.2f}%")

for name, best, cost_fn in [
    ("Bs", best_bs, lambda: cand_rare("key_bs", limit)),
    ("Cs", best_cs, lambda: cand_rare("key_cs", limit)),
    ("Ds", best_ds, lambda: cand_compound("key_ds", 2, 3, limit)),
    ("Es", best_es, lambda: cand_compound("key_es", 4, 6, limit)),
]:
    gain = (word_keys | (best <= limit)).mean() - word_keys.mean()
    print(f"  + {name}:  +{100 * gain:.2f}% recall   costs ~{cost_fn() / 1e6:.1f}M pairs")

word keys: 91.86%
  + Bs:  +0.00% recall   costs ~2.6M pairs
  + Cs:  +0.02% recall   costs ~6.2M pairs
  + Ds:  +0.31% recall   costs ~27.4M pairs
  + Es:  +0.60% recall   costs ~60.1M pairs


In [24]:
for df in [s1, s2, s3]:
    drop = [c for c in df.columns if c.startswith(("key_bs", "key_cs", "key_ds", "key_es"))] + ["name_sound", "addr_sound"]
    df.drop(columns=drop, inplace=True, errors="ignore")
del best_bs, best_cs, best_ds, best_es
import gc; gc.collect()

73

In [25]:
def evaluate(limits):
    la, lb, lc, ld, le = limits["A"], limits["B"], limits["C"], limits["D"], limits["E"]
    fa = (pairs["key_s1"] == pairs["key_other"]) & (pairs["key_s1"] != "") & (pairs["bucket_size"] <= la)
    found = fa | (best_b <= lb) | (best_c <= lc) | (best_d <= ld) | (best_e <= le)

    cost = (s1c["bucket_size"].where(s1c["bucket_size"] <= la, 0).sum()
            + cand_rare("key_b", lb) + cand_rare("key_c", lc)
            + cand_compound("key_d", 2, 3, ld) + cand_compound("key_e", 4, 6, le))
    print(f"{limits}  →  recall {100 * found.mean():.2f}%   pairs ~{cost / 1e6:.0f}M")

evaluate({"A": 50,  "B": 50,  "C": 50,  "D": 50,  "E": 50})
evaluate({"A": 200, "B": 50,  "C": 100, "D": 50,  "E": 50})
evaluate({"A": 200, "B": 100, "C": 200, "D": 50,  "E": 50})
evaluate({"A": 100, "B": 100, "C": 100, "D": 100, "E": 50})

{'A': 50, 'B': 50, 'C': 50, 'D': 50, 'E': 50}  →  recall 91.86%   pairs ~110M
{'A': 200, 'B': 50, 'C': 100, 'D': 50, 'E': 50}  →  recall 93.20%   pairs ~164M
{'A': 200, 'B': 100, 'C': 200, 'D': 50, 'E': 50}  →  recall 93.74%   pairs ~219M
{'A': 100, 'B': 100, 'C': 100, 'D': 100, 'E': 50}  →  recall 92.68%   pairs ~152M


In [26]:
cfg = {"A": 200, "B": 50, "C": 100, "D": 50, "E": 50}

fa = (pairs["key_s1"] == pairs["key_other"]) & (pairs["key_s1"] != "") & (pairs["bucket_size"] <= cfg["A"])
found = (fa | (best_b <= cfg["B"]) | (best_c <= cfg["C"]) |
         (best_d <= cfg["D"]) | (best_e <= cfg["E"]))
print(f"pair recall: {100 * found.mean():.2f}%")

# best possible F0.5 per S1, if the model were perfect on the candidates
per_s1 = pd.DataFrame({"s1": pairs["source1_entity_id"], "found": found}).groupby("s1")["found"].agg(["sum", "count"])
R = per_s1["sum"] / per_s1["count"]
F = 1.25 * R / (0.25 + R)                       # precision = 1 (perfect model)
n_single = gt["matched_entity_ids"].isna().sum()  # singletons always score 1.0
ceiling = (F.sum() + n_single) / len(gt)
print(f"F0.5 ceiling: {100 * ceiling:.2f}%")
print(f"S1 with ZERO matches found: {(per_s1['sum'] == 0).sum():,} of {len(per_s1):,}")

# 25 remaining misses, to see what they look like
miss = pairs[~found].sample(25, random_state=21)
show = pd.DataFrame({
    "name_s1":    miss["source1_entity_id"].map(name_of),
    "name_other": miss["match_id"].map(name_of),
    "addr_s1":    miss["source1_entity_id"].map(addr_of),
    "addr_other": miss["match_id"].map(addr_of),
})
print(show.to_string())

pair recall: 93.20%
F0.5 ceiling: 97.46%
S1 with ZERO matches found: 18,042 of 2,083,574
                                  name_s1                      name_other                                                                                             addr_s1                                                      addr_other
540888                    jay al producer         jy al prdiusar praibhet                              howrah wb marshall house 3rd fl 33 1 ns rd kolkata room no 377 kolkata                                   room no 377 kolkata howrah wb
1236252                   garcia creative                 garcia services                                                                         167 summers st elizabeth wv                                                                
859464                 mumbai plantations                ms mumbai center                                                     105 beur city arman apartment phulwari patna br                        

In [27]:
leet = str.maketrans("0138", "oleb")

def joined_key(name):
    words = [w for w in name.split() if w not in stop_words]
    # digits inside words that also have letters: "audi0" -> "audio"
    words = [w.translate(leet) if any(c.isalpha() for c in w) and any(c.isdigit() for c in w) else w
             for w in words]
    joined = "".join(words)
    return joined[:10] if len(joined) >= 6 else ""      # too short = no key

for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
    df["key_f"] = df["name_core"].map(joined_key)
    print(name, "done")

best_f = smallest_shared_compound("key_fh", s1_combos=[["key_f"]], other_combos=[["key_f"]])

for limit_f in [50, 100, 200]:
    found_f = found | (best_f <= limit_f)
    per_s1 = pd.DataFrame({"s1": pairs["source1_entity_id"], "found": found_f}).groupby("s1")["found"].agg(["sum", "count"])
    R = per_s1["sum"] / per_s1["count"]
    ceiling = (1.25 * R / (0.25 + R)).sum() + n_single
    print(f"F limit {limit_f}:  pair recall {100 * found_f.mean():.2f}%   "
          f"F0.5 ceiling {100 * ceiling / len(gt):.2f}%   "
          f"zero-found S1: {(per_s1['sum'] == 0).sum():,}   "
          f"F pairs ~{cand_compound('key_fh', 1, 1, limit_f) / 1e6:.1f}M")

s1 done
s2 done
s3 done
F limit 50:  pair recall 94.64%   F0.5 ceiling 97.96%   zero-found S1: 15,206   F pairs ~13.4M
F limit 100:  pair recall 94.82%   F0.5 ceiling 98.02%   zero-found S1: 14,776   F pairs ~22.1M
F limit 200:  pair recall 95.29%   F0.5 ceiling 98.24%   zero-found S1: 12,551   F pairs ~59.9M


In [28]:
key_cols = ["entity_id", "country", "key_a", "key_b1", "key_b2", "key_b3",
            "key_c1", "key_c2", "key_c3", "house_no", "key_f"]

s1[key_cols].to_csv("s1_keys.tsv", sep="\t", index=False)
s2[key_cols].to_csv("s2_keys.tsv", sep="\t", index=False)
s3[key_cols].to_csv("s3_keys.tsv", sep="\t", index=False)
print("keys saved!")

keys saved!


In [29]:
import os

for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
    file = f"{name}_keys.tsv"
    size_mb = os.path.getsize(file) / 1024**2
    saved = pd.read_csv(file, sep="\t", dtype=str, keep_default_na=False)

    same_rows = len(saved) == len(df)
    same_ids = (saved["entity_id"].values == df["entity_id"].values).all()
    same_keys = all((saved[c].values == df[c].values).all()
                    for c in ["key_a", "key_b1", "key_c1", "house_no", "key_f"])

    print(f"{file}: {size_mb:.0f} MB, {len(saved):,} rows | "
          f"rows OK: {same_rows} | ids OK: {same_ids} | keys OK: {same_keys}")

print()
print(saved.head(5).to_string())

s1_keys.tsv: 179 MB, 2,206,821 rows | rows OK: True | ids OK: True | keys OK: True
s2_keys.tsv: 405 MB, 5,034,616 rows | rows OK: True | ids OK: True | keys OK: True
s3_keys.tsv: 424 MB, 5,285,603 rows | rows OK: True | ids OK: True | keys OK: True

      entity_id country           key_a   key_b1       key_b2         key_b3   key_c1      key_c2     key_c3 house_no       key_f
0  S3-202863386      US       C500 W416                                          mack      haltom       city           wilfordhan
1  S3-859268022   India  C524 I536 S300    south  consultants  international                                           internatio
2   S3-22467283      US  C536 L655 M523  moncada     learning         center     fawn       worth                5780  moncadalea
3  S3-671162755      US       C100 M520   moynas       coffee                 ivanhoe  cincinnati        box        1  moynascoff
4  S3-960981775   India  E120 P653 V536      efs        print       ventures     41st        22nd  j

In [30]:
import time, gc
import numpy as np
import pandas as pd
from pandas.util import hash_array

s1 = pd.read_csv("s1_keys.tsv", sep="\t", dtype=str, keep_default_na=False)
s2 = pd.read_csv("s2_keys.tsv", sep="\t", dtype=str, keep_default_na=False)
s3 = pd.read_csv("s3_keys.tsv", sep="\t", dtype=str, keep_default_na=False)
print(s1.shape, s2.shape, s3.shape)

(2206821, 11) (5034616, 11) (5285603, 11)


In [31]:
KEYS = {
    #  name: (S1 combos, S2/S3 combos, bucket limit)
    "A": ([["key_a"]], [["key_a"]], 200),
    "B": ([["key_b1"], ["key_b2"]], [["key_b1"], ["key_b2"], ["key_b3"]], 50),
    "C": ([["key_c1"], ["key_c2"]], [["key_c1"], ["key_c2"], ["key_c3"]], 100),
    "D": ([["house_no", "key_c1"], ["house_no", "key_c2"]],
          [["house_no", "key_c1"], ["house_no", "key_c2"], ["house_no", "key_c3"]], 50),
    "E": ([["key_b1", "key_c1"], ["key_b1", "key_c2"], ["key_b2", "key_c1"], ["key_b2", "key_c2"]],
          [["key_b1", "key_c1"], ["key_b1", "key_c2"], ["key_b2", "key_c1"],
           ["key_b2", "key_c2"], ["key_b3", "key_c1"], ["key_b3", "key_c2"]], 50),
    "F": ([["key_f"]], [["key_f"]], 100),
}

def key_hash(df, parts, key_name):
    # joins the parts into one key and turns it into a number; 0 = a part is empty
    key = pd.Series(key_name, index=df.index)
    empty = np.zeros(len(df), dtype=bool)
    for p in parts:
        key = key + "|" + df[p]
        empty |= (df[p] == "").to_numpy()
    h = hash_array(key.to_numpy(dtype=object)).astype(np.int64)
    h[empty] = 0
    return h

def pairs_for_key(a, o, key_name, s1_combos, other_combos, limit):
    # S2/S3 side: (key, row number), then drop buckets bigger than the limit
    o_h = np.concatenate([key_hash(o, p, key_name) for p in other_combos])
    o_j = np.tile(np.arange(len(o), dtype=np.int64), len(other_combos))
    keep = o_h != 0
    other_long = pd.DataFrame({"h": o_h[keep], "j": o_j[keep]}).drop_duplicates()
    size = other_long["h"].value_counts()
    ok = size[size <= limit].index
    other_long = other_long[other_long["h"].isin(ok)]

    # S1 side: (key, row number)
    a_h = np.concatenate([key_hash(a, p, key_name) for p in s1_combos])
    a_i = np.tile(np.arange(len(a), dtype=np.int64), len(s1_combos))
    keep = a_h != 0
    s1_long = pd.DataFrame({"h": a_h[keep], "i": a_i[keep]}).drop_duplicates()
    s1_long = s1_long[s1_long["h"].isin(ok)]

    # same key = candidate pair; store each pair as ONE number: i * (number of others) + j
    m = s1_long.merge(other_long, on="h")
    return np.unique(m["i"].to_numpy() * len(o) + m["j"].to_numpy())

def build_candidates(country):
    a = s1[s1["country"] == country].reset_index(drop=True)
    o = pd.concat([s2[s2["country"] == country], s3[s3["country"] == country]]).reset_index(drop=True)
    print(f"\n{country}: {len(a):,} S1 records, {len(o):,} S2+S3 records")

    all_codes = []
    for name, (s1_combos, other_combos, limit) in KEYS.items():
        start = time.time()
        codes = pairs_for_key(a, o, name, s1_combos, other_combos, limit)
        all_codes.append(codes)
        print(f"  key {name}: {len(codes):>12,} pairs  ({time.time() - start:.0f}s)")

    # combine all keys; n_keys = how many keys found this pair (a useful feature later)
    codes, n_keys = np.unique(np.concatenate(all_codes), return_counts=True)
    del all_codes; gc.collect()
    i = (codes // len(o)).astype(np.int32)
    j = (codes % len(o)).astype(np.int32)
    print(f"  ALL keys: {len(codes):,} different pairs, {len(codes) / len(a):.1f} per S1")
    return a["entity_id"].to_numpy(), o["entity_id"].to_numpy(), i, j, n_keys.astype(np.int8)

In [32]:
for country in ["US", "India"]:
    ids_s1, ids_other, i, j, n_keys = build_candidates(country)
    np.savez(f"candidates_train_{country}.npz",
             ids_s1=ids_s1, ids_other=ids_other, i=i, j=j, n_keys=n_keys)
    print(f"  saved candidates_train_{country}.npz")
    del ids_s1, ids_other, i, j, n_keys
    gc.collect()


US: 1,323,633 S1 records, 6,186,873 S2+S3 records
  key A:   21,626,047 pairs  (22s)
  key B:    4,179,937 pairs  (26s)
  key C:   10,128,109 pairs  (23s)
  key D:   10,924,195 pairs  (57s)
  key E:   25,216,009 pairs  (123s)
  key F:   15,595,495 pairs  (20s)
  ALL keys: 73,712,836 different pairs, 55.7 per S1
  saved candidates_train_US.npz

India: 883,188 S1 records, 4,133,346 S2+S3 records
  key A:   29,298,656 pairs  (18s)
  key B:    3,818,106 pairs  (18s)
  key C:   16,994,147 pairs  (21s)
  key D:    8,278,175 pairs  (41s)
  key E:   11,448,442 pairs  (80s)
  key F:    6,507,657 pairs  (9s)
  ALL keys: 67,635,347 different pairs, 76.6 per S1
  saved candidates_train_India.npz


In [33]:
data_path = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train"
gt = pd.read_csv(data_path + r"\train_ground_truth.tsv", sep="\t", dtype=str)

# true pairs, one row per pair
true = gt.dropna(subset=["matched_entity_ids"]).copy()
true["match_id"] = true["matched_entity_ids"].str.split(",")
true = true.explode("match_id")
true["match_id"] = true["match_id"].str.strip()

found_parts = []
total_pairs = 0
for country in ["US", "India"]:
    d = np.load(f"candidates_train_{country}.npz", allow_pickle=True)
    ids_s1, ids_other, i, j, n_keys = d["ids_s1"], d["ids_other"], d["i"], d["j"], d["n_keys"]
    n_o = len(ids_other)
    codes = i.astype(np.int64) * n_o + j                 # already sorted

    # where each true pair would be in this country's candidate list
    p1 = pd.Index(ids_s1).get_indexer(true["source1_entity_id"])
    p2 = pd.Index(ids_other).get_indexer(true["match_id"])
    here = (p1 >= 0) & (p2 >= 0)
    tcodes = p1[here].astype(np.int64) * n_o + p2[here]
    pos = np.searchsorted(codes, tcodes).clip(max=len(codes) - 1)
    found = codes[pos] == tcodes
    found_parts.append(pd.DataFrame({"s1": true["source1_entity_id"].to_numpy()[here], "found": found}))

    # true pairs vs other pairs: how many keys found them
    is_true = np.zeros(len(codes), dtype=bool)
    is_true[pos[found]] = True
    total_pairs += len(codes)
    per_s1 = np.bincount(i, minlength=len(ids_s1))
    print(f"\n{country}: {len(codes):,} candidate pairs | recall {100 * found.mean():.2f}% | "
          f"per S1: avg {per_s1.mean():.1f}, median {np.median(per_s1):.0f}, max {per_s1.max()}")
    print("  n_keys | true pairs | other pairs")
    for k in range(1, 7):
        print(f"  {k:6d} | {(is_true & (n_keys == k)).sum():>10,} | {(~is_true & (n_keys == k)).sum():>12,}")

# overall recall and F0.5 ceiling
f = pd.concat(found_parts)
per = f.groupby("s1")["found"].agg(["sum", "count"])
R = per["sum"] / per["count"]
n_single = gt["matched_entity_ids"].isna().sum()
ceiling = ((1.25 * R / (0.25 + R)).sum() + n_single) / len(gt)
print(f"\nALL: {total_pairs:,} candidate pairs | pair recall {100 * f['found'].mean():.2f}% | "
      f"F0.5 ceiling {100 * ceiling:.2f}% | S1 with zero found: {(per['sum'] == 0).sum():,}")


US: 73,712,836 candidate pairs | recall 97.16% | per S1: avg 55.7, median 41, max 395
  n_keys | true pairs | other pairs
       1 |    483,654 |   64,960,087
       2 |  1,158,247 |    3,631,079
       3 |  1,207,405 |      572,712
       4 |  1,145,477 |       91,022
       5 |    408,680 |        9,186
       6 |     45,242 |           45

India: 67,635,347 candidate pairs | recall 91.31% | per S1: avg 76.6, median 57, max 418
  n_keys | true pairs | other pairs
       1 |    422,317 |   62,051,265
       2 |    743,030 |    2,375,520
       3 |    693,029 |      315,087
       4 |    553,422 |       79,280
       5 |    316,600 |       18,437
       6 |     65,694 |        1,666

ALL: 141,348,183 candidate pairs | pair recall 94.82% | F0.5 ceiling 98.02% | S1 with zero found: 14,776


In [1]:
import numpy as np, pandas as pd, gc, time
from pandas.util import hash_array

OTHER_COMBOS = {
    "D": [["house_no", "key_c1"], ["house_no", "key_c2"], ["house_no", "key_c3"]],
    "E": [["key_b1", "key_c1"], ["key_b1", "key_c2"], ["key_b2", "key_c1"],
          ["key_b2", "key_c2"], ["key_b3", "key_c1"], ["key_b3", "key_c2"]],
    "F": [["key_f"]],
}

def key_hash(df, parts, key_name):
    key = pd.Series(key_name, index=df.index)
    empty = np.zeros(len(df), dtype=bool)
    for p in parts:
        key = key + "|" + df[p]
        empty |= (df[p] == "").to_numpy()
    h = hash_array(key.to_numpy(dtype=object)).astype(np.int64)
    h[empty] = 0
    return h

def sibling_links(o, keys=("D", "E", "F"), limit=10):
    # pairs of S2/S3 records that share a strong key (small buckets only)
    n_o = len(o)
    all_links = []
    for name in keys:
        h = np.concatenate([key_hash(o, p, name) for p in OTHER_COMBOS[name]])
        jj = np.tile(np.arange(n_o, dtype=np.int64), len(OTHER_COMBOS[name]))
        keep = h != 0
        L = pd.DataFrame({"h": h[keep], "j": jj[keep]}).drop_duplicates()
        size = L["h"].value_counts()
        L = L[L["h"].isin(size[(size >= 2) & (size <= limit)].index)]
        M = L.merge(L, on="h", suffixes=("1", "2"))
        M = M[M["j1"] != M["j2"]]
        all_links.append(np.unique(M["j1"].to_numpy() * n_o + M["j2"].to_numpy()))
        print(f"    links from key {name}: {len(all_links[-1]):,}")
        del L, M; gc.collect()
    return np.unique(np.concatenate(all_links))

def expand(country, s2k, s3k, anchor_min_keys=4, limit=10):
    d = np.load(f"candidates_train_{country}.npz", allow_pickle=True)
    ids_o, i, j, nk = d["ids_other"], d["i"], d["j"], d["n_keys"]
    o = pd.concat([s2k[s2k["country"] == country], s3k[s3k["country"] == country]]).reset_index(drop=True)
    assert (o["entity_id"].to_numpy() == ids_o).all(), "order of S2/S3 records does not match"
    n_o = len(o)
    links = sibling_links(o, limit=limit)
    lj1, lj2 = links // n_o, links % n_o
    anchor = nk >= anchor_min_keys
    new = pd.DataFrame({"i": i[anchor], "j1": j[anchor]}).merge(pd.DataFrame({"j1": lj1, "j2": lj2}), on="j1")
    new_codes = np.unique(new["i"].to_numpy().astype(np.int64) * n_o + new["j2"].to_numpy())
    old_codes = i.astype(np.int64) * n_o + j
    new_codes = new_codes[~np.isin(new_codes, old_codes)]          # only truly new pairs
    return d, old_codes, new_codes, n_o

In [2]:
s2k = pd.read_csv("s2_keys.tsv", sep="\t", dtype=str, keep_default_na=False)
s3k = pd.read_csv("s3_keys.tsv", sep="\t", dtype=str, keep_default_na=False)

data_path = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train"
gt = pd.read_csv(data_path + r"\train_ground_truth.tsv", sep="\t", dtype=str)
true = gt.dropna(subset=["matched_entity_ids"]).copy()
true["match_id"] = true["matched_entity_ids"].str.split(",")
true = true.explode("match_id")
true["match_id"] = true["match_id"].str.strip()

found_parts = []
for country in ["US", "India"]:
    print(country)
    d, old_codes, new_codes, n_o = expand(country, s2k, s3k)
    p1 = pd.Index(d["ids_s1"]).get_indexer(true["source1_entity_id"])
    p2 = pd.Index(d["ids_other"]).get_indexer(true["match_id"])
    here = (p1 >= 0) & (p2 >= 0)
    tcodes = p1[here].astype(np.int64) * n_o + p2[here]
    f_old = np.isin(tcodes, old_codes)
    f_new = np.isin(tcodes, new_codes)
    print(f"  new pairs: {len(new_codes):,} | true matches among them: {f_new.sum():,} | "
          f"recall {100 * f_old.mean():.2f}% -> {100 * (f_old | f_new).mean():.2f}%")
    found_parts.append(pd.DataFrame({"s1": true["source1_entity_id"].to_numpy()[here], "found": f_old | f_new}))
    np.savez(f"expanded_train_{country}.npz",
             i=(new_codes // n_o).astype(np.int32), j=(new_codes % n_o).astype(np.int32))
    del d, old_codes, new_codes; gc.collect()

f = pd.concat(found_parts)
per = f.groupby("s1")["found"].agg(["sum", "count"])
R = per["sum"] / per["count"]
n_single = gt["matched_entity_ids"].isna().sum()
ceiling = ((1.25 * R / (0.25 + R)).sum() + n_single) / len(gt)
print(f"\nALL: pair recall {100 * f['found'].mean():.2f}% (was 94.82%) | "
      f"F0.5 ceiling {100 * ceiling:.2f}% (was 98.02%) | zero-found S1: {(per['sum'] == 0).sum():,} (was 14,776)")

US
    links from key D: 18,567,606
    links from key E: 32,670,664
    links from key F: 8,271,574
  new pairs: 1,622,375 | true matches among them: 2,470 | recall 97.16% -> 97.22%
India
    links from key D: 12,029,366
    links from key E: 18,905,842
    links from key F: 5,548,754
  new pairs: 991,567 | true matches among them: 4,767 | recall 91.31% -> 91.47%

ALL: pair recall 94.92% (was 94.82%) | F0.5 ceiling 98.04% (was 98.02%) | zero-found S1: 14,768 (was 14,776)


In [3]:
import re, numpy as np, pandas as pd
from collections import Counter, defaultdict
from anyascii import anyascii
from rapidfuzz import process, fuzz

cols = ["entity_id", "business_name", "name_clean"]
names = pd.concat([pd.read_csv(f"{s}_clean.tsv", sep="\t", dtype=str, keep_default_na=False, usecols=cols)
                   for s in ["s1", "s2", "s3"]]).set_index("entity_id")
names["is_hindi"] = names["business_name"].str.contains(r"[\u0900-\u097F]", regex=True)
print("records:", len(names), " hindi names:", names["is_hindi"].sum())

data_path = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train"
gt = pd.read_csv(data_path + r"\train_ground_truth.tsv", sep="\t", dtype=str)
true = gt.dropna(subset=["matched_entity_ids"]).copy()
true["match_id"] = true["matched_entity_ids"].str.split(",")
true = true.explode("match_id")
true["match_id"] = true["match_id"].str.strip()

records: 12527040  hindi names: 427427


In [4]:
def hindi_to_english_letters(text):
    return anyascii(text.replace("ं", "न्").replace("ॉ", "ो"))

def basic_clean(col):
    # same steps as our training cleaning
    col = col.fillna("").astype(str).apply(hindi_to_english_letters).str.lower()
    col = col.str.replace("&", " and ", regex=False).str.replace("'", "", regex=False)
    col = col.str.replace(r"[^a-z0-9 ]", " ", regex=True)
    col = col.str.replace(r"\b(\w+)( \1\b)+", r"\1", regex=True)
    col = col.str.replace(r"\s+", " ", regex=True).str.strip()
    return col.str.replace(r"\b([a-z]) (?=[a-z]\b)", r"\1", regex=True)

same_word = {"limited": "ltd", "private": "pvt", "corporation": "corp", "company": "co", "incorporated": "inc"}
def normalize(t):
    return " ".join(same_word.get(w, w) for w in t.split())

def learn_hindi_map(pairs, min_count=3, min_share=0.6):
    word_pairs = Counter()
    for eng, hin in zip(pairs["english"], pairs["hindi"]):
        e, h = normalize(eng).split(), normalize(hin).split()
        if len(e) != len(h):
            continue                                  # same position = same word only if lengths match
        for hw, ew in zip(h, e):
            if hw != ew:
                word_pairs[(hw, ew)] += 1
    options = defaultdict(Counter)
    for (hw, ew), c in word_pairs.items():
        options[hw][ew] += c
    m = {}
    for hw, cnt in options.items():
        ew, c = cnt.most_common(1)[0]
        if c >= min_count and c / sum(cnt.values()) >= min_share:
            m[hw] = ew
    return m

def apply_map(col, m):
    return pd.Series([" ".join(m.get(w, w) for w in s.split()) for s in col], index=col.index)

In [5]:
hindi_ids = set(names.index[names["is_hindi"]])

# every record of a business: the S1 plus all its matches
groups = pd.concat([
    pd.DataFrame({"s1": true["source1_entity_id"].unique(), "rec": true["source1_entity_id"].unique()}),
    pd.DataFrame({"s1": true["source1_entity_id"], "rec": true["match_id"]}),
])
h = groups[groups["rec"].isin(hindi_ids)].rename(columns={"rec": "hindi_id"})
e = groups[~groups["rec"].isin(hindi_ids)].rename(columns={"rec": "eng_id"})
pairs = h.merge(e, on="s1")                            # each Hindi record × each English record of the same business
pairs["hindi"] = basic_clean(names.loc[pairs["hindi_id"], "business_name"]).to_numpy()
pairs["english"] = names.loc[pairs["eng_id"], "name_clean"].to_numpy()
print("learning pairs:", len(pairs), " businesses:", pairs["s1"].nunique())

learning pairs: 840332  businesses: 154981


In [6]:
rng = np.random.default_rng(0)
s1s = pairs["s1"].unique()
learn_s1 = set(rng.choice(s1s, len(s1s) // 2, replace=False))
learn, check = pairs[pairs["s1"].isin(learn_s1)], pairs[~pairs["s1"].isin(learn_s1)]

def similarity(hindi_col, m):
    h = apply_map(hindi_col, m) if m else hindi_col
    return process.cpdist(check["english"].map(normalize).tolist(), h.map(normalize).tolist(),
                          scorer=fuzz.token_set_ratio, workers=-1).mean()

print(f"no word list:         avg similarity {similarity(check['hindi'], None):.1f}")
for mc in [5, 3, 2]:
    m = learn_hindi_map(learn, min_count=mc)
    print(f"min_count {mc}: {len(m):5d} words -> avg similarity {similarity(check['hindi'], m):.1f}")

no word list:         avg similarity 67.0
min_count 5:   129 words -> avg similarity 93.8
min_count 3:   129 words -> avg similarity 93.8
min_count 2:   129 words -> avg similarity 93.8


In [9]:
# H5 - save the learned hindi words to the txt file
path = "dictionaries/hindi_words.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write("# Hindi (transliterated) -> English words, learned from train pairs\n")
    f.write("# format: hindi spelling <TAB> english\n\n")
    for h, e in sorted(m.items()):
        f.write(f"{h}\t{e}\n")

print("saved", len(m), "words to", path)

saved 129 words to dictionaries/hindi_words.txt


In [10]:
# F1 - count S1 records that got ZERO candidates from our buckets
import os, glob, csv
import numpy as np
import pandas as pd

train_dir = r"C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train"

# ground truth file (finds it by name)
gt_path = glob.glob(os.path.join(train_dir, "*ground*"))[0]
gt = pd.read_csv(gt_path, sep="\t", dtype=str, keep_default_na=False)
gt = gt.set_index("source1_entity_id")["matched_entity_ids"]
print("ground truth:", gt_path)

# which S2/S3 records are written in hindi
hindi_ids = set()
for s in ["s2", "s3"]:
    df = pd.read_csv(f"{s}_clean.tsv", sep="\t", dtype=str, keep_default_na=False,
                     usecols=["entity_id", "business_name"], quoting=csv.QUOTE_NONE)
    is_hindi = df["business_name"].str.contains(r"[ऀ-ॿ]", regex=True)
    hindi_ids.update(df.loc[is_hindi, "entity_id"])
print("hindi records in s2+s3:", len(hindi_ids))

zero_ids = []   # save them for the next step
for country in ["US", "India"]:
    d = np.load(f"candidates_train_{country}.npz", allow_pickle=True)
    ids_s1 = d["ids_s1"]
    has_cand = np.zeros(len(ids_s1), dtype=bool)
    has_cand[d["i"]] = True
    zero = ids_s1[~has_cand]
    zero = [x for x in zero if x in gt.index]
    zero_ids += zero

    matches = gt.loc[zero]
    singleton = (matches == "").sum()
    real = matches[matches != ""]
    with_hindi = real.apply(lambda m: any(x in hindi_ids for x in m.split(","))).sum()

    print(f"\n--- {country} ---")
    print(f"S1 total:                    {len(ids_s1):,}")
    print(f"S1 with ZERO candidates:     {len(zero):,}  ({100*len(zero)/len(ids_s1):.2f}%)")
    print(f"   true singletons (fine):   {singleton:,}")
    print(f"   have real match (LOST):   {len(real):,}")
    print(f"      match is hindi:        {with_hindi:,}")
    print(f"      match is english:      {len(real) - with_hindi:,}")

ground truth: C:\Users\Hp\OneDrive\Desktop\amazon hackthon\student_resource\dataset\train\train_ground_truth.tsv
hindi records in s2+s3: 427427

--- US ---
S1 total:                    1,323,633
S1 with ZERO candidates:     312  (0.02%)
   true singletons (fine):   73
   have real match (LOST):   239
      match is hindi:        0
      match is english:      239

--- India ---
S1 total:                    883,188
S1 with ZERO candidates:     622  (0.07%)
   true singletons (fine):   53
   have real match (LOST):   569
      match is hindi:        331
      match is english:      238


In [11]:
# F2 - zero-candidate check per country on TEST
import glob
import numpy as np

files = sorted(glob.glob("candidates_test_*.npz"))
if len(files) == 0:
    print("no candidates_test files here - run this in the folder where run_all was run")

for path in files:
    country = path.replace("candidates_test_", "").replace(".npz", "")
    d = np.load(path, allow_pickle=True)
    n_s1 = len(d["ids_s1"])
    cand_per_s1 = np.bincount(d["i"], minlength=n_s1)
    zero = (cand_per_s1 == 0).sum()

    print(f"--- {country} ---")
    print(f"S1 total:                {n_s1:,}")
    print(f"S1 with ZERO candidates: {zero:,}  ({100*zero/n_s1:.2f}%)")
    print(f"avg candidates per S1:   {cand_per_s1.mean():.1f}\n")

--- France ---
S1 total:                259,452
S1 with ZERO candidates: 289  (0.11%)
avg candidates per S1:   76.0

--- India ---
S1 total:                809,986
S1 with ZERO candidates: 775  (0.10%)
avg candidates per S1:   77.6

--- US ---
S1 total:                663,106
S1 with ZERO candidates: 72  (0.01%)
avg candidates per S1:   52.2

